# Preparação do Dataset

## Configurações Iniciais

### Importar bibliotecas

In [ ]:
import pandas as pd

### Carregar datasets

Carregar datasets de português e matemática e avaliar duplicatas e dados faltantes

In [ ]:
df_mat = pd.read_csv("../dataset/student-mat.csv", sep=";")
df_por = pd.read_csv("../dataset/student-por.csv", sep=";")

def avaliar_dataframe(df:pd.DataFrame):
    display(df)
    print(f"Há duplicatas: {df.duplicated().any()}")
    print(f"Há dados faltantes: {df.isnull().any().any()}")

avaliar_dataframe(df_mat)
avaliar_dataframe(df_por)

Não há dados faltantes ou duplicatas (tratamento feito no dataset antes da nossa aquisição).

## Merge dos Datasets

### Merge das tabelas

Operação realizada conforme as orientações disponibilizadas com o dataset. A mesma é apresentada em dataset/student-merge.R
Serão mantidos apenas registros com todas as informações, pois desejamos avaliar e comparar o desempenho em ambas.

In [ ]:
merge_cols = ["school","sex","age","address","famsize","Pstatus", "Medu","Fedu","Mjob","Fjob","reason","nursery","internet"]
df_merge = pd.merge(df_mat, df_por, on=merge_cols, suffixes=("_mat","_por"))

display(df_merge)
print(df_merge.columns)

## Investigar e Tratar Inconsistências

### Comparar colunas "equivalentes"

Os campos inclusos no parâmetro 'on' da função 'merge' são específicos do aluno (como 'school', 'sex', 'age', 'address', etc), já aqueles que ficaram de fora deveriam ser específicas da disciplina ('absences', 'G1', 'G2', etc). No entanto, percebemos algumas colunas que deveriam ser únicas para o aluno, mas que após a operação estão separadas por disciplina (tais como 'guardian', 'traveltime', 'famrel', etc). Então, é necessário investigar as diferenças entre esses campos.

In [ ]:
def comparar_colunas_equivalentes(df:pd.DataFrame):
    sufixo_colunas_duplicadas = [col.replace("_mat", "") for col in df.columns if col.endswith("_mat")]
    
    diffs = []
    for col in sufixo_colunas_duplicadas:
        num_diffs = (df[f"{col}_mat"] != df[f"{col}_por"]).sum()
        diffs.append({
            "Coluna": col,
            "Diferenças": num_diffs
        })

    df_diffs = pd.DataFrame(diffs).sort_values(by="Diferenças")
    display(df_diffs)

comparar_colunas_equivalentes(df_merge)

As colunas se separaram claramente entre aquelas que acreditávamos ser unicas do estudante (<15 diferenças, sendo provavelmente inconsistências) e aquelas específicas da disciplina (>50 diferenças).

### Tratar inconsistencias

Observar o número de registros com inconsistências

In [ ]:
def filtrar_registros_concordantes(df:pd.DataFrame):
    filtros = (
        (df["schoolsup_por"] == df["schoolsup_mat"]) &
        (df["higher_por"] == df["higher_mat"]) &
        (df["famsup_por"] == df["famsup_mat"]) &
        (df["traveltime_por"] == df["traveltime_mat"]) &
        (df["Dalc_por"] == df["Dalc_mat"]) &
        (df["activities_por"] == df["activities_mat"]) &
        (df["guardian_por"] == df["guardian_mat"]) &
        (df["romantic_por"] == df["romantic_mat"]) &
        (df["health_por"] == df["health_mat"]) &
        (df["studytime_por"] == df["studytime_mat"]) &
        (df["goout_por"] == df["goout_mat"]) &
        (df["famrel_por"] == df["famrel_mat"]) &
        (df["freetime_por"] == df["freetime_mat"]) &
        (df["Walc_por"] == df["Walc_mat"])
    )
    return df.loc[filtros].reset_index(drop=True)

df_filter = filtrar_registros_concordantes(df_merge)
display(df_filter)

Apenas 12 dos 382 registros foram removidos após trarar essas inconsistências.

### Remover colunas redundantes

Várias das colunas que agora são tratadas como específicas dos estudantes são iguais, devendo então ser removidas.

In [ ]:
def remover_colunas_redundantes(df:pd.DataFrame):
    cols_redundantes = ["schoolsup_por", "higher_por", "famsup_por", "traveltime_por", "Dalc_por", "activities_por", "guardian_por", "romantic_por", "health_por", "studytime_por", "goout_por", "famrel_por", "freetime_por", "Walc_por"]
    df = df.drop(columns=cols_redundantes)

    map_novos_nomes = {col.replace("_por", "_mat"): col.replace("_por", "") for col in cols_redundantes}
    df = df.rename(columns=map_novos_nomes)

    return df

df_clean = remover_colunas_redundantes(df_filter)
display(df_clean)
print(df_clean.columns)


## Renomear Valores e Atributos

Tornar dataset mais fácil de ser interpretado

### Função Genérica para renomear coluna e seus valores

In [ ]:
def ajustar_coluna(df:pd.DataFrame, coluna_atual, novo_nome=None, mapa_valores=None):
    # Renomeia a coluna (se necessário)
    if novo_nome:
        df = df.rename(columns={coluna_atual: novo_nome})
        coluna = novo_nome
    else:
        coluna = coluna_atual
    
    # Aplica transformação de valores (se houver)
    if mapa_valores:
        df[coluna] = df[coluna].replace(mapa_valores)
    
    return df

### Ajustar cada Coluna

Alterar nome da coluna o fazer o map de seus valores. 

In [ ]:
df_renamed:pd.DataFrame = df_clean.copy()

df_renamed = (
    df_renamed
    .pipe(ajustar_coluna, "school", "NOME_ESCOLA", {
        "GP": "Gabriel Pereira",
        "MS": "Mousinho da Silveira"
    })
    .pipe(ajustar_coluna, "sex", "SEXO", {
        "F": "feminino",
        "M": "masculino"
    })
    .pipe(ajustar_coluna, "age", "IDADE")

    .pipe(ajustar_coluna, "address", "TIPO_ENDERECO", {
        "U": "urbano",
        "R": "rural"
    })
    .pipe(ajustar_coluna, "famsize", "TAMANHO_FAMILIA", {
        "LE3": "ate 3",
        "GT3": "mais de 3"
    })
    .pipe(ajustar_coluna, "Pstatus", "STATUS_PAIS", {
        "T": "juntos",
        "A": "separados"
    })

    # 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education -> não alterar para manter como ordinal
    .pipe(ajustar_coluna, "Medu", "ESCOLARIDADE_MAE")
    .pipe(ajustar_coluna, "Fedu", "ESCOLARIDADE_PAI")

    .pipe(ajustar_coluna, "Mjob", "PROFISSAO_MAE", {
        "teacher": "professor",
        "health": "saude",
        "services": "servicos",
        "at_home": "casa",
        "other": "outro"
    })
    .pipe(ajustar_coluna, "Fjob", "PROFISSAO_PAI", {
        "teacher": "professor",
        "health": "saude",
        "services": "servicos",
        "at_home": "casa",
        "other": "outro"
    })

    .pipe(ajustar_coluna, "reason", "RAZAO_ESCOLHA_ESCOLA", {
        "home": "proximo_casa",
        "reputation": "reputacao",
        "course": "curso",
        "other": "outro"
    })
    .pipe(ajustar_coluna, "guardian", "RESPONSAVEL", {
        "mother": "mae",
        "father": "pai",
        "other": "outro"
    })

    #1 - <15 min., 2 - 15 to 30 min., 3 - 30 min. to 1 hour, or 4 - >1 hour -> Não alterar para manter ordinal
    .pipe(ajustar_coluna, "traveltime", "TEMPO_DESLOCAMENTO")

    #1 - <2 hours, 2 - 2 to 5 hours, 3 - 5 to 10 hours, or 4 - >10 hours) 15 failures - number of past class failures (numeric: n if 1<=n<3, else 4 -> não alterar para manter ordinal
    .pipe(ajustar_coluna, "studytime", "TEMPO_ESTUDO_SEMANAL")

    # 4 significa 4 ou mais -> não alterar para manter ordinal
    .pipe(ajustar_coluna, "failures_mat", "NUM_REPROVACOES_MAT")
    .pipe(ajustar_coluna, "failures_por", "NUM_REPROVACOES_POR")

    .pipe(ajustar_coluna, "schoolsup", "SUPORTE_ESCOLAR", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "famsup", "SUPORTE_FAMILIAR", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "paid_mat", "AULAS_PAGAS_MAT", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "paid_por", "AULAS_PAGAS_POR", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "activities", "ATIVIDADES_EXTRAS", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "nursery", "CRECHE", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "higher", "ALMEJA_ENSINO_SUPERIOR", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "internet", "INTERNET_CASA", {"yes": True, "no": False})
    .pipe(ajustar_coluna, "romantic", "RELACIONAMENTO", {"yes": True, "no": False})

    # Não alterar pois usam escalas likert (ordinais)
    .pipe(ajustar_coluna, "famrel", "QUALIDADE_FAMILA")
    .pipe(ajustar_coluna, "freetime", "TEMPO_LIVRE")
    .pipe(ajustar_coluna, "goout", "SAIR_COM_AMIGOS")
    .pipe(ajustar_coluna, "Dalc", "ALCOOL_DIA_UTIL")
    .pipe(ajustar_coluna, "Walc", "ALCOOL_FIM_DE_SEMANA")
    .pipe(ajustar_coluna, "health", "SAUDE")

    # Outros
    .pipe(ajustar_coluna, "absences_mat", "FALTAS_MAT")
    .pipe(ajustar_coluna, "absences_por", "FALTAS_POR")

    # Notas
    .pipe(ajustar_coluna, "G1_mat", "NOTA_1_MAT")
    .pipe(ajustar_coluna, "G2_mat", "NOTA_2_MAT")
    .pipe(ajustar_coluna, "G3_mat", "NOTA_3_MAT")
    .pipe(ajustar_coluna, "G1_por", "NOTA_1_POR")
    .pipe(ajustar_coluna, "G2_por", "NOTA_2_POR")
    .pipe(ajustar_coluna, "G3_por", "NOTA_3_POR")
)

display(df_renamed)
print(df_renamed.columns)

### Organizar as colunas

Alterar a ordem das colunas para agrupalas de modo mais intuitivo

In [ ]:
df_renamed = df_renamed[[
    'SEXO',
    'IDADE',
    'CRECHE',
    'SAUDE',

    'NOME_ESCOLA',
    'RAZAO_ESCOLHA_ESCOLA',
    'TEMPO_ESTUDO_SEMANAL',
    'ALMEJA_ENSINO_SUPERIOR',
    'SUPORTE_ESCOLAR',

    'RESPONSAVEL',
    'STATUS_PAIS',
    'ESCOLARIDADE_MAE',
    'PROFISSAO_MAE',
    'ESCOLARIDADE_PAI',
    'PROFISSAO_PAI',
    'TAMANHO_FAMILIA',
    'QUALIDADE_FAMILA',
    'SUPORTE_FAMILIAR',

    'INTERNET_CASA',
    'TIPO_ENDERECO',
    'TEMPO_DESLOCAMENTO',

    'ATIVIDADES_EXTRAS',
    'RELACIONAMENTO',
    'TEMPO_LIVRE',
    'SAIR_COM_AMIGOS',
    'ALCOOL_DIA_UTIL',
    'ALCOOL_FIM_DE_SEMANA',

    'AULAS_PAGAS_MAT',
    'NUM_REPROVACOES_MAT',
    'FALTAS_MAT',
    'NOTA_1_MAT',
    'NOTA_2_MAT',
    'NOTA_3_MAT',

    'AULAS_PAGAS_POR',
    'NUM_REPROVACOES_POR',
    'FALTAS_POR',
    'NOTA_1_POR',
    'NOTA_2_POR',
    'NOTA_3_POR'
]]

display(df_renamed)
print(df_renamed.columns)

df_renamed.to_csv("../dataset/student.csv", index=False)